# Lecture 6 — Tutorial 3.29: Applied Data Quality and Cleaning Problem Solving

This notebook contains the **worked solutions** for the ten applied problems shown on the tutorial website. Try each problem from the website first. The solution cells are intentionally detailed and repeat imports so that each problem can be studied independently.

The datasets are synthetic teaching data. They do not describe real municipalities, organisations, staff or service users.

## Recommended use

1. Read the problem on the tutorial website.
2. Make your own audit/plan before opening the solution.
3. Run the solution from a fresh runtime.
4. Compare the reasoning, not only the final output.
5. Change one rule or column and observe what changes.

**Numbering rule:** each problem keeps the same identifier used on the tutorial website. This notebook does not use a separate local problem numbering scheme.

## Problem 3.29.1 — Build a defensible quality audit for municipal service requests

The aim is to inspect the file before changing it. The solution deliberately separates exact duplicates from repeated identifiers and creates an issue register.

In [ ]:
# Import pandas for table-shaped data.
import pandas as pd

# Point to the raw Dataset A file in the course repository.
url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_raw.csv"

# Load the original file and immediately create a working copy.
raw = pd.read_csv(url)
df = raw.copy()

# Confirm the dataset dimensions.
print("Rows and columns:", df.shape)

# Build a compact column-level audit table.
audit = pd.DataFrame({
    "column": df.columns,
    "dtype_detected": [str(df[c].dtype) for c in df.columns],
    "non_missing": [df[c].notna().sum() for c in df.columns],
    "missing": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
})
print("\nCOLUMN AUDIT")
print(audit.to_string(index=False))

# Exact duplicate rows are different from repeated identifiers.
exact_duplicates = df.duplicated().sum()
repeated_ids = df["request_id"].duplicated(keep=False).sum()
print("\nExact duplicate rows:", exact_duplicates)
print("Rows involved in repeated request_id values:", repeated_ids)

# Inspect key categorical columns without dropping missing values.
category_columns = ["city", "service_type", "channel", "status", "priority", "repeat_contact"]
for column in category_columns:
    print(f"\nVALUE COUNTS: {column}")
    print(df[column].value_counts(dropna=False))

# Test date conversion without changing the raw columns.
opened_test = pd.to_datetime(df["opened_date"], errors="coerce")
closed_test = pd.to_datetime(df["closed_date"], errors="coerce")
print("\nUnusable opened_date values:")
print(df.loc[opened_test.isna() & df["opened_date"].notna(), ["request_id", "opened_date"]])
print("\nUnusable closed_date values:")
print(df.loc[closed_test.isna() & df["closed_date"].notna(), ["request_id", "closed_date"]])

# Test intended numeric columns safely.
resolution_test = pd.to_numeric(df["resolution_hours"], errors="coerce")
satisfaction_test = pd.to_numeric(df["satisfaction_score"], errors="coerce")
print("\nNon-numeric resolution_hours entries:")
print(df.loc[resolution_test.isna() & df["resolution_hours"].notna(), ["request_id", "resolution_hours"]])
print("\nNon-numeric satisfaction_score entries:")
print(df.loc[satisfaction_test.isna() & df["satisfaction_score"].notna(), ["request_id", "satisfaction_score"]])

# Create a starter issue register. Students can extend it after inspecting outputs.
issue_register = pd.DataFrame([
    ["Exact duplicate rows", int(exact_duplicates), "clear error", "Check whether duplicate ingestion occurred"],
    ["Repeated request identifiers", int(repeated_ids), "possible logical contradiction", "Inspect records sharing the same request_id"],
    ["Invalid opening dates", int(opened_test.isna().sum()), "clear error / missing", "Cannot support reliable time analysis"],
    ["Non-numeric resolution values", int(resolution_test.isna().sum()), "clear error / missing", "Use safe conversion and retain flag"],
    ["Satisfaction outside 1–5", int(((satisfaction_test < 1) | (satisfaction_test > 5)).sum()), "clear error", "Flag rather than silently recode"],
    ["City spelling/spacing variants", None, "likely formatting inconsistency", "Standardise only obvious equivalents"],
    ["Service/channel variants", None, "likely formatting inconsistency", "Use explicit mapping"],
    ["Closed cases without a usable close date", None, "possible logical contradiction", "Review status/date relationship"],
], columns=["issue", "count_or_note", "classification", "recommended_response"])
print("\nSTARTER ISSUE REGISTER")
print(issue_register.to_string(index=False))

## Problem 3.29.2 — Clean dates, numbers and logical contradictions without hiding uncertainty

This solution shows a transparent structural cleaning pipeline. It removes only confirmed exact duplicates and creates flags for uncertainty.

In [ ]:
# Import pandas for loading, cleaning and validating the CSV.
import pandas as pd

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_raw.csv"
raw = pd.read_csv(url)
df = raw.copy()

# Keep the original row count for the transformation log.
raw_rows = len(df)

# Remove only exact duplicate rows.
df = df.drop_duplicates().copy()
removed_exact_duplicates = raw_rows - len(df)

# Standardise obvious city variants using an explicit mapping.
city_map = {
    "copenhagen": "Copenhagen", "cph": "Copenhagen",
    "aalborg": "Aalborg", "ålborg": "Aalborg",
    "odense": "Odense", "aarhus": "Aarhus", "århus": "Aarhus",
}
df["city_clean"] = df["city"].astype("string").str.strip().str.lower().map(city_map)

# Standardise service-type variants with an explicit mapping.
service_map = {
    "housing advice": "Housing Advice",
    "waste collection": "Waste Collection",
    "citizen id": "Citizen ID",
    "citizen-id": "Citizen ID",
    "parking": "Parking",
    "family support": "Family Support",
}
df["service_type_clean"] = df["service_type"].astype("string").str.strip().str.lower().map(service_map)

# Standardise channel variants.
channel_map = {"digital": "Digital", "phone": "Phone", "in person": "In Person", "in-person": "In Person"}
df["channel_clean"] = df["channel"].astype("string").str.strip().str.lower().map(channel_map)

# Parse dates safely. Unusable values become NaT, not invented dates.
df["opened_date_clean"] = pd.to_datetime(df["opened_date"], errors="coerce")
df["closed_date_clean"] = pd.to_datetime(df["closed_date"], errors="coerce")

# Convert intended numeric columns safely.
df["resolution_hours_clean"] = pd.to_numeric(df["resolution_hours"], errors="coerce")
df["satisfaction_score_clean"] = pd.to_numeric(df["satisfaction_score"], errors="coerce")

# Standardise Yes/No variants while keeping the raw repeat_contact column.
repeat_map = {"yes": "Yes", "y": "Yes", "1": "Yes", "true": "Yes",
              "no": "No", "n": "No", "0": "No", "false": "No"}
df["repeat_contact_clean"] = df["repeat_contact"].astype("string").str.strip().str.lower().map(repeat_map)

# Create separate, readable quality flags.
df["flag_invalid_open_date"] = df["opened_date_clean"].isna()
df["flag_closed_before_open"] = (
    df["opened_date_clean"].notna() &
    df["closed_date_clean"].notna() &
    (df["closed_date_clean"] < df["opened_date_clean"])
)
df["flag_closed_missing_date"] = df["status"].eq("Closed") & df["closed_date_clean"].isna()
df["flag_invalid_resolution"] = df["resolution_hours_clean"].isna() | (df["resolution_hours_clean"] < 0)
df["flag_invalid_satisfaction"] = (
    df["satisfaction_score_clean"].notna() &
    ~df["satisfaction_score_clean"].between(1, 5)
)
df["flag_repeated_id"] = df["request_id"].duplicated(keep=False)

# Combine individual flags while preserving each one.
flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["any_quality_issue"] = df[flag_cols].any(axis=1)

# Derive month only from a valid parsed opening date.
df["report_month"] = df["opened_date_clean"].dt.to_period("M").astype("string")

# Compare selected quality indicators before and after transformation.
print("Raw rows:", raw_rows)
print("Rows after exact duplicate removal:", len(df))
print("Exact duplicates removed:", removed_exact_duplicates)
print("Rows with at least one quality flag:", int(df["any_quality_issue"].sum()))
print("\nClean city labels:", sorted(df["city_clean"].dropna().unique()))
print("Clean service labels:", sorted(df["service_type_clean"].dropna().unique()))
print("Clean channel labels:", sorted(df["channel_clean"].dropna().unique()))

# Build a transformation log from actual operations.
transformation_log = pd.DataFrame([
    [1, "all columns", "Removed confirmed exact duplicate rows", "Exact copies add no new observation", removed_exact_duplicates],
    [2, "city", "Created city_clean with explicit mapping", "Make obvious label variants comparable", int((df["city"].astype(str).str.strip() != df["city_clean"].astype(str)).sum())],
    [3, "service_type", "Created service_type_clean", "Standardise obvious variants", int(df["service_type_clean"].notna().sum())],
    [4, "channel", "Created channel_clean", "Standardise channel variants", int(df["channel_clean"].notna().sum())],
    [5, "opened_date / closed_date", "Parsed dates with errors='coerce'", "Reveal invalid date values without inventing replacements", int(df["opened_date_clean"].isna().sum() + df["closed_date_clean"].isna().sum())],
    [6, "resolution_hours / satisfaction_score", "Converted to numeric with errors='coerce'", "Support range checks", int(df["flag_invalid_resolution"].sum() + df["flag_invalid_satisfaction"].sum())],
    [7, "quality flags", "Created individual flags and any_quality_issue", "Preserve uncertainty for later analytical decisions", int(df["any_quality_issue"].sum())],
], columns=["step", "variable", "action", "rationale", "rows_affected_or_flagged"])
print("\nTRANSFORMATION LOG")
print(transformation_log.to_string(index=False))

# Inspect repeated IDs rather than deleting them automatically.
print("\nREPEATED REQUEST IDS FOR MANUAL REVIEW")
print(df.loc[df["flag_repeated_id"], ["request_id", "city", "service_type", "opened_date", "status"]].sort_values("request_id"))

## Problem 3.29.3 — Prepare service feedback for later text analysis

The raw text remains untouched. The solution creates a separate prepared field and preserves negation words.

In [ ]:
# Import pandas for the table and NLTK for basic text preparation.
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download the small NLTK resources required in a fresh Colab runtime.
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_raw.csv"
df = pd.read_csv(url).drop_duplicates().copy()

# Preserve the original feedback explicitly.
df["feedback_raw"] = df["feedback"]

# Missing feedback becomes an empty processing string, not a claim about sentiment.
df["feedback_safe"] = df["feedback_raw"].fillna("").astype(str)

# Build a stop-word set but preserve negation words because they can reverse meaning.
custom_stopwords = set(stopwords.words("english")) - {"not", "no", "nor"}

def prepare_text(text):
    # Lowercase before tokenisation so "Wait" and "wait" are comparable.
    text = text.lower()
    # Tokenise the text into individual word/punctuation units.
    tokens = word_tokenize(text)
    # Keep alphabetic tokens only and remove ordinary stop words.
    cleaned = [token for token in tokens if token.isalpha() and token not in custom_stopwords]
    return cleaned

# Apply the function and also create a convenient joined-text version.
df["feedback_tokens"] = df["feedback_safe"].apply(prepare_text)
df["feedback_clean"] = df["feedback_tokens"].apply(lambda tokens: " ".join(tokens))
df["flag_missing_feedback"] = df["feedback_safe"].str.strip().eq("")

# Demonstrate why negation matters.
demo = "The wait was not long."
naive_stopwords = set(stopwords.words("english"))
naive = [w for w in word_tokenize(demo.lower()) if w.isalpha() and w not in naive_stopwords]
careful = prepare_text(demo)
print("Original:", demo)
print("Naive stop-word removal:", naive)
print("Negation-preserving removal:", careful)

# Inspect original and prepared text side by side.
print("\nORIGINAL VS PREPARED FEEDBACK")
print(df[["request_id", "feedback_raw", "feedback_clean", "flag_missing_feedback"]].head(15).to_string(index=False))
print("\nMissing/empty feedback rows:", int(df["flag_missing_feedback"].sum()))

## Problem 3.29.4 — Audit community digital-support sessions

The audit pattern is transferred to a new schema rather than copied blindly.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_raw.csv"
raw = pd.read_csv(url)
df = raw.copy()

print("Rows and columns:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))
print("\nExact duplicates:", int(df.duplicated().sum()))
print("Rows in repeated session_id groups:", int(df["session_id"].duplicated(keep=False).sum()))

for column in ["city", "centre", "support_topic", "age_band", "access_mode", "resolved", "follow_up_needed", "staff_role"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

# Safe type tests reveal values that cannot be interpreted normally.
date_test = pd.to_datetime(df["session_date"], errors="coerce")
wait_test = pd.to_numeric(df["wait_minutes"], errors="coerce")
duration_test = pd.to_numeric(df["session_minutes"], errors="coerce")
sat_test = pd.to_numeric(df["satisfaction_score"], errors="coerce")

print("\nInvalid dates:")
print(df.loc[date_test.isna(), ["session_id", "session_date"]])
print("\nNegative/non-numeric waits:")
print(df.loc[wait_test.isna() | (wait_test < 0), ["session_id", "wait_minutes"]])
print("\nNon-positive or >240 minute sessions (teaching threshold):")
print(df.loc[duration_test.isna() | (duration_test <= 0) | (duration_test > 240), ["session_id", "session_minutes"]])
print("\nInvalid satisfaction:")
print(df.loc[sat_test.notna() & ~sat_test.between(1,5), ["session_id", "satisfaction_score"]])

# Inspect potentially contradictory resolution/follow-up combinations.
print("\nResolution/follow-up combinations:")
print(pd.crosstab(df["resolved"], df["follow_up_needed"], dropna=False))

## Problem 3.29.5 — Build a transparent cleaning pipeline for support sessions

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_raw.csv"
raw = pd.read_csv(url)
df = raw.drop_duplicates().copy()

# Explicit mappings for obvious formatting variants.
city_map = {"copenhagen":"Copenhagen", "aalborg":"Aalborg", "ålborg":"Aalborg", "odense":"Odense", "aarhus":"Aarhus", "århus":"Aarhus"}
centre_map = {"library hub":"Library Hub", "citizen centre":"Citizen Centre", "community lab":"Community Lab", "mobile desk":"Mobile Desk"}
topic_map = {"digital id":"Digital ID", "online benefits":"Online Benefits", "job portal":"Job Portal", "health booking":"Health Booking", "email & documents":"Email & Documents", "e-mail & documents":"Email & Documents"}
age_map = {"18–29":"18–29", "30–44":"30–44", "30-44":"30–44", "45–59":"45–59", "60+":"60+", "60 plus":"60+"}
access_map = {"own device":"Own device", "centre device":"Centre device", "centre-device":"Centre device", "phone support":"Phone support"}
yes_no_map = {"yes":"Yes", "y":"Yes", "1":"Yes", "true":"Yes", "resolved":"Yes", "needed":"Yes", "no":"No", "n":"No", "0":"No", "false":"No", "not resolved":"No", "not needed":"No"}

df["city_clean"] = df["city"].astype("string").str.strip().str.lower().map(city_map)
df["centre_clean"] = df["centre"].astype("string").str.strip().str.lower().map(centre_map)
df["support_topic_clean"] = df["support_topic"].astype("string").str.strip().str.lower().map(topic_map)
df["age_band_clean"] = df["age_band"].astype("string").str.strip().map(age_map)
df["access_mode_clean"] = df["access_mode"].astype("string").str.strip().str.lower().map(access_map)
df["resolved_clean"] = df["resolved"].astype("string").str.strip().str.lower().map(yes_no_map)
df["follow_up_needed_clean"] = df["follow_up_needed"].astype("string").str.strip().str.lower().map(yes_no_map)

# Parse dates and derive a reporting month.
df["session_date_clean"] = pd.to_datetime(df["session_date"], errors="coerce")
df["report_month"] = df["session_date_clean"].dt.to_period("M").astype("string")

# Convert numeric values.
df["wait_minutes_clean"] = pd.to_numeric(df["wait_minutes"], errors="coerce")
df["session_minutes_clean"] = pd.to_numeric(df["session_minutes"], errors="coerce")
df["satisfaction_score_clean"] = pd.to_numeric(df["satisfaction_score"], errors="coerce")

# Quality flags keep uncertainty visible.
df["flag_invalid_date"] = df["session_date_clean"].isna()
df["flag_invalid_wait"] = df["wait_minutes_clean"].isna() | (df["wait_minutes_clean"] < 0)
df["flag_invalid_duration"] = df["session_minutes_clean"].isna() | (df["session_minutes_clean"] <= 0) | (df["session_minutes_clean"] > 240)
df["flag_invalid_satisfaction"] = df["satisfaction_score_clean"].notna() & ~df["satisfaction_score_clean"].between(1,5)
df["flag_followup_contradiction"] = df["resolved_clean"].eq("Yes") & df["follow_up_needed_clean"].eq("Yes")
df["flag_repeated_id"] = df["session_id"].duplicated(keep=False)
flag_cols=[c for c in df.columns if c.startswith("flag_")]
df["any_quality_issue"] = df[flag_cols].any(axis=1)

print("Raw rows:", len(raw))
print("After exact duplicate removal:", len(df))
print("Flagged rows:", int(df["any_quality_issue"].sum()))
print("\nResolved/follow-up clean cross-tab:")
print(pd.crosstab(df["resolved_clean"], df["follow_up_needed_clean"], dropna=False))

## Problem 3.29.6 — Prepare support-session notes and document what the cleaning cannot tell you

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_raw.csv"
df = pd.read_csv(url).drop_duplicates().copy()

df["notes_raw"] = df["notes"]
df["notes_safe"] = df["notes_raw"].fillna("").astype(str)
custom_stopwords = set(stopwords.words("english")) - {"not", "no", "nor"}

def prepare_notes(text):
    tokens = word_tokenize(text.lower())
    return [token for token in tokens if token.isalpha() and token not in custom_stopwords]

df["notes_tokens"] = df["notes_safe"].apply(prepare_notes)
df["notes_clean"] = df["notes_tokens"].apply(lambda x: " ".join(x))
df["flag_missing_notes"] = df["notes_safe"].str.strip().eq("")

# Sample across several topics rather than only the first rows.
sample = (
    df.groupby("support_topic", group_keys=False)
      .head(3)
      [["session_id", "support_topic", "notes_raw", "notes_clean"]]
)
print(sample.to_string(index=False))
print("\nMissing notes:", int(df["flag_missing_notes"].sum()))

## Problem 3.29.7 — Audit an organisational automation pilot

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/automation_pilot_raw.csv"
df = pd.read_csv(url)
print("Rows and columns:", df.shape)
print("Exact duplicates:", int(df.duplicated().sum()))
print("Rows in repeated task_id groups:", int(df["task_id"].duplicated(keep=False).sum()))
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False))

for column in ["city", "organisation_unit", "task_type", "execution_mode", "human_review", "outcome", "staff_sentiment", "tool_version"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

# Safe conversion checks.
date_test = pd.to_datetime(df["task_date"], errors="coerce")
minutes = pd.to_numeric(df["minutes_spent"], errors="coerce")
errors = pd.to_numeric(df["errors_found"], errors="coerce")
confidence = pd.to_numeric(df["confidence_rating"], errors="coerce")

print("\nInvalid dates:\n", df.loc[date_test.isna(), ["task_id", "task_date"]])
print("\nInvalid minutes:\n", df.loc[minutes.isna() | (minutes < 0), ["task_id", "minutes_spent"]])
print("\nInvalid errors:\n", df.loc[errors.isna() | (errors < 0), ["task_id", "errors_found"]])
print("\nInvalid confidence:\n", df.loc[confidence.notna() & ~confidence.between(1,5), ["task_id", "confidence_rating"]])

# High-error records without a conventional Yes review label deserve inspection.
high_error_no_review = df[(errors >= 5) & (df["human_review"].astype(str).str.strip().str.lower().isin(["no","n","0","false"]))]
print("\nHigh-error / no-review cases:\n", high_error_no_review[["task_id","execution_mode","errors_found","human_review","outcome"]])

## Problem 3.29.8 — Clean task records while preserving evaluation-relevant exceptions

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/automation_pilot_raw.csv"
raw = pd.read_csv(url)
df = raw.drop_duplicates().copy()

city_map = {"copenhagen":"Copenhagen", "aalborg":"Aalborg", "ålborg":"Aalborg", "odense":"Odense", "aarhus":"Aarhus", "århus":"Aarhus"}
unit_map = {x.lower(): x for x in ["Finance","Citizen Services","HR","Procurement","Communications"]}
task_map = {x.lower(): x for x in ["Data Entry","Document Check","Scheduling","Status Update","Email Triage"]}
mode_map = {"manual":"Manual", "automated":"Automated", "auto":"Automated", "ai-assisted":"AI-assisted", "ai assisted":"AI-assisted"}
sentiment_map = {"positive":"Positive", "neutral":"Neutral", "concerned":"Concerned"}
yes_no = {"yes":"Yes","y":"Yes","1":"Yes","true":"Yes","no":"No","n":"No","0":"No","false":"No"}

df["city_clean"] = df["city"].astype("string").str.strip().str.lower().map(city_map)
df["organisation_unit_clean"] = df["organisation_unit"].astype("string").str.strip().str.lower().map(unit_map)
df["task_type_clean"] = df["task_type"].astype("string").str.strip().str.lower().map(task_map)
df["execution_mode_clean"] = df["execution_mode"].astype("string").str.strip().str.lower().map(mode_map)
df["staff_sentiment_clean"] = df["staff_sentiment"].astype("string").str.strip().str.lower().map(sentiment_map)
df["human_review_clean"] = df["human_review"].astype("string").str.strip().str.lower().map(yes_no)

df["task_date_clean"] = pd.to_datetime(df["task_date"], errors="coerce")
df["report_month"] = df["task_date_clean"].dt.to_period("M").astype("string")
df["minutes_spent_clean"] = pd.to_numeric(df["minutes_spent"], errors="coerce")
df["errors_found_clean"] = pd.to_numeric(df["errors_found"], errors="coerce")
df["confidence_rating_clean"] = pd.to_numeric(df["confidence_rating"], errors="coerce")

df["flag_invalid_date"] = df["task_date_clean"].isna()
df["flag_invalid_minutes"] = df["minutes_spent_clean"].isna() | (df["minutes_spent_clean"] < 0)
df["flag_invalid_errors"] = df["errors_found_clean"].isna() | (df["errors_found_clean"] < 0)
df["flag_invalid_confidence"] = df["confidence_rating_clean"].notna() & ~df["confidence_rating_clean"].between(1,5)
df["flag_high_error_no_review"] = (df["errors_found_clean"] >= 5) & df["human_review_clean"].eq("No")
df["flag_repeated_id"] = df["task_id"].duplicated(keep=False)
flag_cols=[c for c in df.columns if c.startswith("flag_")]
df["any_quality_issue"] = df[flag_cols].any(axis=1)

print("Raw rows:", len(raw))
print("Rows after exact duplicate removal:", len(df))
print("Rows with any quality issue:", int(df["any_quality_issue"].sum()))
print("\nExecution mode before:\n", raw["execution_mode"].value_counts(dropna=False))
print("\nExecution mode after:\n", df["execution_mode_clean"].value_counts(dropna=False))
print("\nHigh-error / no-review cases:\n", df.loc[df["flag_high_error_no_review"], ["task_id","task_type_clean","execution_mode_clean","errors_found_clean","outcome"]])

## Problem 3.29.9 — Prepare staff comments for responsible later analysis

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/automation_pilot_raw.csv"
df = pd.read_csv(url).drop_duplicates().copy()
df["comment_raw"] = df["comment"]
df["comment_safe"] = df["comment_raw"].fillna("").astype(str)
custom_stopwords = set(stopwords.words("english")) - {"not", "no", "nor"}

def prepare_comment(text):
    tokens = word_tokenize(text.lower())
    return [token for token in tokens if token.isalpha() and token not in custom_stopwords]

df["comment_tokens"] = df["comment_safe"].apply(prepare_comment)
df["comment_clean"] = df["comment_tokens"].apply(lambda x: " ".join(x))
df["flag_missing_comment"] = df["comment_safe"].str.strip().eq("")

print(df[["task_id","execution_mode","comment_raw","comment_clean"]].head(15).to_string(index=False))

# Retrieve comments containing selected governance-relevant words.
terms = ["not", "review", "error", "trust", "failed", "human", "faster"]
pattern = "|".join(terms)
examples = df[df["comment_safe"].str.lower().str.contains(pattern, regex=True, na=False)]
print("\nSELECTED ORIGINAL COMMENTS")
print(examples[["task_id","execution_mode","staff_sentiment","comment_raw"]].head(25).to_string(index=False))

## Problem 3.29.10 — Prepare two datasets for a responsible cross-dataset comparison

The key lesson is to aggregate each event-level dataset to a compatible city-month unit **before** merging.

In [ ]:
import pandas as pd

# Use the curated files after you have attempted your own cleaning.
service_url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_cleaned.csv"
support_url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_cleaned.csv"
service = pd.read_csv(service_url)
support = pd.read_csv(support_url)

# Convert the cleaned numeric columns because CSV reloads may infer mixed types.
service["satisfaction_score_clean"] = pd.to_numeric(service["satisfaction_score_clean"], errors="coerce")
support["satisfaction_score_clean"] = pd.to_numeric(support["satisfaction_score_clean"], errors="coerce")
support["wait_minutes_clean"] = pd.to_numeric(support["wait_minutes_clean"], errors="coerce")

# Dataset A: create one row per city + month.
service_summary = (
    service[service["report_month"].notna() & service["city"].notna()]
    .groupby(["city", "report_month"], as_index=False)
    .agg(
        request_count=("request_id", "count"),
        mean_satisfaction=("satisfaction_score_clean", "mean"),
        valid_satisfaction_n=("satisfaction_score_clean", "count"),
    )
)

# Dataset B: create the SAME unit of observation before merging.
support_summary = (
    support[support["report_month"].notna() & support["city"].notna()]
    .groupby(["city", "report_month"], as_index=False)
    .agg(
        support_session_count=("session_id", "count"),
        median_wait_minutes=("wait_minutes_clean", "median"),
        mean_support_satisfaction=("satisfaction_score_clean", "mean"),
    )
)

# Verify uniqueness of the proposed join keys.
print("Duplicate city-month keys in service summary:", int(service_summary.duplicated(["city","report_month"]).sum()))
print("Duplicate city-month keys in support summary:", int(support_summary.duplicated(["city","report_month"]).sum()))

# Merge the aggregate tables, not the original event-level rows.
combined = service_summary.merge(
    support_summary,
    on=["city", "report_month"],
    how="outer",
    indicator=True,
)

print("\nMERGE STATUS")
print(combined["_merge"].value_counts())
print("\nCOMBINED CITY-MONTH TABLE")
print(combined.head(20).to_string(index=False))

# Demonstrate the many-to-many danger without actually performing the huge direct merge.
service_key_counts = service.groupby(["city","report_month"]).size().rename("service_rows")
support_key_counts = support.groupby(["city","report_month"]).size().rename("support_rows")
join_risk = pd.concat([service_key_counts, support_key_counts], axis=1).fillna(0)
join_risk["potential_direct_pairs"] = join_risk["service_rows"] * join_risk["support_rows"]
print("\nPotential row multiplication from an event-level city-month merge:")
print(join_risk.sort_values("potential_direct_pairs", ascending=False).head(10))